# Pharmaceutical ML Analysis using Databricks and Spark

This notebook was recovered from a Databricks HTML export.

## Project Summary
This project creates and analyzes a 10,000-sample pharmaceutical dataset to explore how solubility and permeability relate to drug absorption. It uses Spark-based data processing, clustering, visualization, and predictive modeling.

## Key Methods
- PySpark data generation and preprocessing
- Feature categorization
- KMeans clustering
- Exploratory visualization
- Linear regression
- Decision tree regression
- Model evaluation using RMSE


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
from pyspark.sql.functions import udf, when

# Create SparkSession
spark = SparkSession.builder \
    .appName("DrugSampleGenerator") \
    .getOrCreate()

# Define schema for drug samples
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("solubility", DoubleType(), True),
    StructField("permeability", DoubleType(), True)
])

# Generate 10,000 drug samples with random solubility and permeability values
drug_samples_data = [(i, random.uniform(0.1, 1.0), random.uniform(0.1, 1.0)) for i in range(10000)]

# Create DataFrame
drug_samples_df = spark.createDataFrame(drug_samples_data, schema)

# Define function to categorize solubility and permeability
def categorize(value):
    if 0.01 <= value <= 0.5:
        return "Low"
    elif 0.6 <= value <= 1.0:
        return "High"
    else:
        return None

# Define UDF for categorization
categorize_udf = udf(categorize)

# Categorize solubility and permeability
drug_samples_categorized_df = drug_samples_df.withColumn("solubility_category", categorize_udf("solubility")) \
    .withColumn("permeability_category", categorize_udf("permeability"))

# Show DataFrame
display(drug_samples_categorized_df)


In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
import random

# Convert DataFrame to features vector
assembler = VectorAssembler(
    inputCols=["solubility", "permeability"],
    outputCol="features")

features_df = assembler.transform(drug_samples_categorized_df.select("solubility", "permeability"))

# Train KMeans model
kmeans = KMeans(k=4, seed=1)
model = kmeans.fit(features_df)

# Add cluster labels to DataFrame
clustered_df = model.transform(features_df).select("solubility", "permeability", "prediction")

# Display clustered DataFrame
display(clustered_df)


In [ ]:
import matplotlib.pyplot as plt

# Convert DataFrame to Pandas DataFrame
pandas_df = drug_samples_categorized_df.select("solubility", "permeability").toPandas()

# Plot histogram for solubility
plt.figure(figsize=(10, 5))
plt.hist(pandas_df['solubility'], bins=20, color='blue', alpha=0.7)
plt.xlabel('Solubility')
plt.ylabel('Frequency')
plt.title('Histogram of Solubility')
plt.grid(True)
plt.show()

# Plot histogram for permeability
plt.figure(figsize=(10, 5))
plt.hist(pandas_df['permeability'], bins=20, color='green', alpha=0.7)
plt.xlabel('Permeability')
plt.ylabel('Frequency')
plt.title('Histogram of Permeability')
plt.grid(True)
plt.show()


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
from pyspark.sql.functions import udf, when

# Create SparkSession
spark = SparkSession.builder \
    .appName("AbsorptionDataset") \
    .getOrCreate()

# Define schema for absorption dataset
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("absorption", DoubleType(), True)
])

# Generate 10,000 drug samples with absorption values ranging from 0.15 to 0.95
absorption_data = [(i, round(random.uniform(0.15, 0.95), 2)) for i in range(10000)]

# Create DataFrame
absorption_df = spark.createDataFrame(absorption_data, schema)

# Define function to categorize absorption
def categorize_absorption(value):
    if 0.15 <= value <= 0.5:
        return "Low"
    elif 0.6 <= value <= 0.95:
        return "High"
    else:
        return None

# Define UDF for categorization
categorize_absorption_udf = udf(categorize_absorption)

# Categorize absorption
absorption_categorized_df = absorption_df.withColumn("absorption_category", categorize_absorption_udf("absorption"))

# Show DataFrame
display(absorption_categorized_df)


In [ ]:
# Merge the absorption dataset with the solubility and permeability dataset
merged_df = absorption_categorized_df.crossJoin(drug_samples_categorized_df)

# Show the merged DataFrame
display(merged_df)


In [ ]:
# Merge the absorption dataset with the solubility and permeability dataset
merged_df = absorption_categorized_df.join(drug_samples_categorized_df, "id")

# Show the merged DataFrame
display(merged_df)


In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Combine features into a single feature vector
assembler = VectorAssembler(
    inputCols=["solubility", "permeability"],
    outputCol="features")

# Apply the vector assembler to the dataset
feature_df = assembler.transform(merged_df)

# Select the required columns
final_df = feature_df.select("features", "absorption")

# Split the data into training and test sets
(train_data, test_data) = final_df.randomSplit([0.8, 0.2], seed=1234)

# Train the linear regression model
lr = LinearRegression(featuresCol="features", labelCol="absorption")
model = lr.fit(train_data)

# Make predictions on the test data
predictions = model.transform(test_data)

# Evaluate the model
evaluator = RegressionEvaluator(labelCol="absorption", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print("Root Mean Squared Error (RMSE):", rmse)


In [ ]:
#The results indicate that the linear regression model has a Root Mean Squared Error (RMSE) of approximately 0.227.

#RMSE is a measure of how well the model's predictions match the actual values. It represents the square root of the average squared difference between the predicted and actual values. In this case, an RMSE of 0.227 indicates that, on average, the model's predictions deviate from the actual absorption values by approximately 0.227 units.

#Lower values of RMSE indicate better model performance. Therefore, an RMSE of 0.227 suggests that the model is making reasonably accurate predictions, with relatively small errors.

#However, the interpretation of the RMSE value depends on the scale of the target variable (absorption). If the scale of the target variable is small, an RMSE of 0.227 might be considered acceptable. But if the scale of the target variable is large, the RMSE should be interpreted relative to the scale of the target variable.

#Overall, a RMSE of 0.227 suggests that the linear regression model is performing reasonably well in predicting absorption based on solubility and permeability variables.

In [ ]:
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Train the decision tree regression model
dt = DecisionTreeRegressor(featuresCol="features", labelCol="absorption")
dt_model = dt.fit(train_data)

# Make predictions on the test data
dt_predictions = dt_model.transform(test_data)

# Evaluate the model
dt_evaluator = RegressionEvaluator(labelCol="absorption", predictionCol="prediction", metricName="rmse")
dt_rmse = dt_evaluator.evaluate(dt_predictions)
print("Decision Tree RMSE:", dt_rmse)



In [ ]:
##The decision tree model has a Root Mean Squared Error (RMSE) of approximately 0.234.

#RMSE is a measure of how well the model's predictions match the actual values. It represents the square root of the average squared difference between the predicted and actual values. In this case, an RMSE of 0.234 indicates that, on average, the model's predictions deviate from the actual absorption values by approximately 0.234 units.

#As with the linear regression model, lower values of RMSE indicate better model performance. Therefore, an RMSE of 0.234 suggests that the decision tree model is making reasonably accurate predictions, with relatively small errors.

#Similarly, the interpretation of the RMSE value depends on the scale of the target variable (absorption). If the scale of the target variable is small, an RMSE of 0.234 might be considered acceptable. But if the scale of the target variable is large, the RMSE should be interpreted relative to the scale of the target variable.

#Overall, an RMSE of 0.234 suggests that the decision tree model is performing reasonably well in predicting absorption based on solubility and permeability variables. Comparing with the linear regression model, both models seem to have similar performance in terms of RMSE.





In [ ]:
from sklearn.tree import export_text
import matplotlib.pyplot as plt

# Extract decision tree rules
tree_rules = export_text(dt_model, feature_names=["solubility", "permeability"])

# Plot decision tree rules
plt.figure(figsize=(10, 6))
plt.text(0.1, 0.5, tree_rules, fontsize=12)
plt.axis('off')
plt.show()
